# Dynamic transition rates example

This notebook introduces dynamic `transition_rates` callbacks. Instead of supplying one fixed matrix, we pass a function that receives the current `SimulationState` and returns the transition-rate matrix for the next simulation step.

The example uses a three-state progression `0 -> 1 -> 2`. The `0 -> 1` rate turns on **over time** (i.e. explicit time-dependence), while the `1 -> 2` rate slows down as the **number of cells in state 2 increases** (i.e. non-linear feedback, leading to implicit time-dependence).

## Preparations

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from markovmodus import SimulationParameters, SimulationState, simulate_dataset

In [ ]:
# Helper functions.
def num_states_from_adata(adata) -> int:
    return int(np.asarray(adata.uns["state_expression"]).shape[0])


def state_palette(num_states: int) -> list:
    cmap = plt.get_cmap("tab10")
    return [cmap(state % cmap.N) for state in range(num_states)]


def plot_transition_graph(
    matrix: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    positions: np.ndarray | None = None,
    palette: list | None = None,
    state_names: list[str] | None = None,
) -> None:
    matrix = np.asarray(matrix, dtype=float)
    num_states = matrix.shape[0]
    if positions is None:
        angles = np.linspace(0, 2 * np.pi, num_states, endpoint=False)
        positions = np.stack((np.cos(angles), np.sin(angles)), axis=1)
    else:
        positions = np.asarray(positions, dtype=float)

    max_rate = float(matrix.max())
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_aspect("equal")
    margin = 0.45
    ax.set_xlim(float(positions[:, 0].min() - margin), float(positions[:, 0].max() + margin))
    ax.set_ylim(float(positions[:, 1].min() - margin), float(positions[:, 1].max() + margin))
    ax.axis("off")

    node_radius = 0.12
    if palette is None:
        palette = state_palette(num_states)
    if state_names is None:
        state_names = [str(state) for state in range(num_states)]
    edge_color = "0.58"

    if max_rate > 0:
        for source in range(num_states):
            for target in range(num_states):
                rate = matrix[source, target]
                if source == target or rate <= 0:
                    continue

                source_pos = positions[source]
                target_pos = positions[target]
                bidirectional = matrix[target, source] > 0
                rad = 0.20 if bidirectional else 0.03
                linewidth = 1.4 + 1.8 * (rate / max_rate)

                ax.annotate(
                    "",
                    xy=target_pos,
                    xytext=source_pos,
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=edge_color,
                        alpha=0.85,
                        lw=linewidth,
                        mutation_scale=14,
                        shrinkA=18,
                        shrinkB=18,
                        connectionstyle=f"arc3,rad={rad}",
                    ),
                    zorder=1,
                )

                direction = target_pos - source_pos
                normal = np.array([-direction[1], direction[0]])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                label_position = 0.55 * source_pos + 0.45 * target_pos + normal * rad * 0.55
                ax.text(
                    label_position[0],
                    label_position[1],
                    f"{rate:.2f}",
                    fontsize=8,
                    color="0.35",
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.8),
                    zorder=2,
                )

    for state, (x, y) in enumerate(positions):
        node = plt.Circle(
            (x, y),
            node_radius,
            facecolor=palette[state % len(palette)],
            edgecolor="black",
            linewidth=1.2,
            alpha=0.85,
            zorder=3,
        )
        ax.add_patch(node)
        ax.text(
            x,
            y,
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=11,
            weight="bold",
            zorder=4,
        )


def preprocess_for_scanpy(adata):
    adata.obs["state"] = adata.obs["state"].astype("category")
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    adata.layers["log_normalized"] = adata.X.copy()

    analysis = adata.copy()
    sc.pp.scale(analysis, max_value=10)
    sc.pp.neighbors(analysis, n_neighbors=min(15, analysis.n_obs - 1), use_rep="X")
    sc.tl.umap(analysis)
    adata.obsm["X_umap"] = analysis.obsm["X_umap"].copy()
    return adata


def state_counts(adata) -> pd.Series:
    counts = adata.obs["state"].astype(int).value_counts().reindex(
        range(num_states_from_adata(adata)), fill_value=0
    )
    counts.index.name = "state"
    return counts


def plot_state_counts(
    adata,
    title: str,
    *,
    ax: plt.Axes,
    palette: list | None = None,
    state_names: list[str] | None = None,
) -> None:
    counts = state_counts(adata)
    if palette is None:
        palette = state_palette(num_states_from_adata(adata))
    if state_names is None:
        state_names = [str(state) for state in counts.index]
    bar_colors = [palette[int(state) % len(palette)] for state in counts.index]
    counts.plot(kind="bar", ax=ax, color=bar_colors)
    ax.set_xlabel("State")
    ax.set_ylabel("Cells")
    ax.set_xticklabels(state_names, rotation=0)
    ax.set_title(title)


def plot_embedding_with_state_labels(
    adata,
    coordinates: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    palette: list,
    xlabel: str,
    ylabel: str,
    state_names: list[str] | None = None,
) -> None:
    states = adata.obs["state"].astype(int).to_numpy()
    if state_names is None:
        state_names = [str(state) for state in range(num_states_from_adata(adata))]
    for state in range(num_states_from_adata(adata)):
        mask = states == state
        if not mask.any():
            continue
        color = palette[state % len(palette)]
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=18,
            alpha=1.0,
            color=color,
            linewidths=0,
            zorder=1,
        )
        center = coordinates[mask].mean(axis=0)
        ax.scatter(
            center[0],
            center[1],
            s=520,
            color=color,
            edgecolor="black",
            linewidth=1.2,
            alpha=0.7,
            zorder=3,
        )
        ax.text(
            center[0],
            center[1],
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=10,
            weight="bold",
            zorder=4,
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


## Define a dynamic transition model

For dynamic transitions, `transition_rates` is a function that has the signature `Callable[[SimulationState], np.ndarray]`. The part in square brackets, `[SimulationState]`, is the input: the callback receives one `SimulationState` instance at the start of each simulation step. The return annotation after the comma, `np.ndarray`, is the output: the callback must return the transition-rate matrix for that step.

For a state-level transition model, the returned array has shape `(num_states, num_states)`. Rows are source states, columns are destination states, and off-diagonal entries are continuous-time rates. Diagonal entries are ignored and set to zero internally.

The current `SimulationState` exposes:

- `time`: the current simulation time.
- `cell_states`: the current latent state of every cell.
- `unspliced` and `spliced`: the current U/S count matrices.
- `birth_parent` and `birth_time`: row-creation lineage metadata, mainly useful when proliferation is enabled.
- `generation` and `last_division_time`: division-history metadata, mainly useful when proliferation is enabled.

This means transition rates can depend on external time, current state occupancy, current expression levels, or lineage history. In this example, the callback uses two of those pieces of state:

- `state.time`: an external induction signal increases the `0 -> 1` rate over time.
- `state.cell_states`: the current number of state-2 cells feeds back on the `1 -> 2` rate.

In [ ]:
NUM_STATES = 3
NUM_GENES = 90
STATE_NAMES = ["0", "1", "2"]
STATE_POSITIONS = np.array([[-1.0, 0.0], [0.0, 0.0], [1.0, 0.0]])
STATE_COLORS = state_palette(NUM_STATES)
TIMEPOINTS = [4.0, 8.0, 12.0, 16.0]
TIMEPOINT_LABELS = [f"t = {timepoint:g}" for timepoint in TIMEPOINTS]


def sigmoid(x: float, *, midpoint: float, scale: float) -> float:
    return 1.0 / (1.0 + np.exp(-(x - midpoint) / scale))


def dynamic_transition_rates(state: SimulationState) -> np.ndarray:
    terminal_fraction = np.mean(state.cell_states == 2) if state.cell_states.size else 0.0
    induction = sigmoid(state.time, midpoint=6.0, scale=1.3)
    density_feedback = 1.0 / (1.0 + (terminal_fraction / 0.35) ** 3)

    transition_rates = np.zeros((NUM_STATES, NUM_STATES), dtype=float)
    transition_rates[0, 1] = 0.03 + 0.18 * induction
    transition_rates[1, 2] = 0.20 * density_feedback
    return transition_rates


def make_params(t_final: float) -> SimulationParameters:
    return SimulationParameters(
        # Latent dynamics.
        num_states=NUM_STATES,
        transition_rates=dynamic_transition_rates,

        # State-specific spliced-count targets and U/S dynamics.
        num_genes=NUM_GENES,
        markers_per_state=20,
        marker_overlap={(0, 1): 4, (1, 2): 4},
        baseline_expression=0.25,
        marker_expression=12.0,
        splicing_rate=1.0,
        decay_rate=1.0,

        # Initial condition.
        initial_state_probs=[1.0, 0.0, 0.0],

        # Simulation size, time, and reproducibility.
        num_cells=500,
        t_final=t_final,
        dt=0.5,
        rng_seed=22,
    )


params = make_params(max(TIMEPOINTS))

## Inspect the transition rate functions

Below we show how the transition rate function behaves for different `SimulationState` inputs (synthetic example states).
More precisely, we look at the transition graph early, after the induction signal has turned on, and after many cells have accumulated in state 2.

In [ ]:
def synthetic_state(time: float, state_counts: list[int]) -> SimulationState:
    states = np.repeat(np.arange(NUM_STATES), state_counts).astype(int)
    zeros = np.zeros((states.size, NUM_GENES), dtype=int)
    return SimulationState(
        time=time,
        cell_states=states,
        unspliced=zeros,
        spliced=zeros,
        birth_parent=np.full(states.size, -1, dtype=int),
        birth_time=np.zeros(states.size, dtype=float),
        generation=np.zeros(states.size, dtype=int),
        last_division_time=np.zeros(states.size, dtype=float),
    )


scenarios = [
    ("early", synthetic_state(0.0, [500, 0, 0])),
    ("induced", synthetic_state(8.0, [250, 250, 0])),
    ("state 2 filled", synthetic_state(14.0, [100, 150, 250])),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (title, state) in zip(axes, scenarios):
    plot_transition_graph(
        dynamic_transition_rates(state),
        title,
        ax=ax,
        positions=STATE_POSITIONS,
        palette=STATE_COLORS,
        state_names=STATE_NAMES,
    )
plt.tight_layout()

In [ ]:
times = np.linspace(0.0, params.t_final, 200)
terminal_fractions = np.linspace(0.0, 0.8, 200)

rate_0_to_1 = [
    dynamic_transition_rates(synthetic_state(time, [500, 0, 0]))[0, 1]
    for time in times
]
rate_1_to_2 = [
    dynamic_transition_rates(
        synthetic_state(12.0, [0, int(500 * (1 - fraction)), int(500 * fraction)])
    )[1, 2]
    for fraction in terminal_fractions
]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(times, rate_0_to_1)
axes[0].set_xlabel("time")
axes[0].set_ylabel("0 -> 1 rate")
axes[0].set_title("time-dependent induction")

axes[1].plot(terminal_fractions, rate_1_to_2)
axes[1].set_xlabel("fraction in state 2")
axes[1].set_ylabel("1 -> 2 rate")
axes[1].set_title("population feedback")
plt.tight_layout()

## Simulate several final-time datasets

`markovmodus` currently returns the final state of a simulation, not the full trajectory. To inspect the dynamic system at several times, we run the same configuration repeatedly with different `t_final` values.

The datasets are concatenated before Scanpy preprocessing so that the UMAP panels use one shared embedding space.

In [ ]:
timepoint_adatas = [simulate_dataset(make_params(timepoint)) for timepoint in TIMEPOINTS]
combined_adata = ad.concat(
    timepoint_adatas,
    label="timepoint",
    keys=TIMEPOINT_LABELS,
    index_unique="-",
)
combined_adata.obs["timepoint"] = pd.Categorical(
    combined_adata.obs["timepoint"],
    categories=TIMEPOINT_LABELS,
    ordered=True,
)
combined_adata.uns["state_expression"] = np.asarray(params.state_expression).tolist()
combined_adata

## Inspect the generated data

We use total-count normalization, log transformation, nearest-neighbor graph construction, and UMAP. The raw U/S counts remain available in `combined_adata.layers["unspliced"]` and `combined_adata.layers["spliced"]`.

In [ ]:
combined_adata = preprocess_for_scanpy(combined_adata)
timepoint_views = {
    label: combined_adata[combined_adata.obs["timepoint"] == label].copy()
    for label in TIMEPOINT_LABELS
}
combined_adata

In [ ]:
fig, axes = plt.subplots(
    2,
    len(TIMEPOINT_LABELS),
    figsize=(4.2 * len(TIMEPOINT_LABELS), 7.2),
    gridspec_kw={"height_ratios": [2.1, 1.0]},
)
max_population = max(state_counts(view).max() for view in timepoint_views.values())

for column, label in enumerate(TIMEPOINT_LABELS):
    view = timepoint_views[label]
    plot_embedding_with_state_labels(
        view,
        view.obsm["X_umap"],
        f"UMAP at {label}",
        ax=axes[0, column],
        palette=STATE_COLORS,
        xlabel="UMAP1",
        ylabel="UMAP2",
        state_names=STATE_NAMES,
    )
    plot_state_counts(
        view,
        f"State populations at {label}",
        ax=axes[1, column],
        palette=STATE_COLORS,
        state_names=STATE_NAMES,
    )
    axes[1, column].set_ylim(0, max_population * 1.1)

plt.tight_layout()